# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sandesh30-cloud/FlyRank-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np, os
from datasets import load_dataset
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
all_files = api.list_repo_files('FlyRank/internship-warehouse', repo_type='dataset')
march_files = [f for f in all_files if 'fact_content_daily_performance' in f
               and 'sample' not in f and '2026-03' in f]
if not march_files:
    print('[WARNING] No files matched - inspect all_files and fix the filter:')
    for f in [x for x in all_files if 'fact_content_daily_performance' in x and 'sample' not in x][:20]:
        print(' ', f)
    raise ValueError('Fix march_files filter above, then re-run.')

dataset = load_dataset('FlyRank/internship-warehouse', data_files={'train': march_files}, split='train')
needed_cols = ['report_date','client_hash_id','content_hash_id',
               'gsc_data_available','gsc_impressions','gsc_clicks','gsc_avg_position']
cols_present = [c for c in needed_cols if c in dataset.column_names]
dataset = dataset.select_columns(cols_present)
raw = dataset.to_pandas()
raw['report_date'] = pd.to_datetime(raw['report_date'])
raw['day_of_month'] = raw['report_date'].dt.day
raw = raw[raw['gsc_data_available'] == True].copy()
print(f'Loaded {len(raw)} GSC-available rows for March 2026.')



## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
first_half = raw[raw['day_of_month'] <= 15]
second_half = raw[raw['day_of_month'] > 15]

# --- Engineered numeric features (days 1-15 only) ---
feat = (first_half.groupby(['client_hash_id','content_hash_id'])
        .agg(clicks_first_half=('gsc_clicks','sum'),
             impressions_first_half=('gsc_impressions','sum'),
             days_with_position_fh=('gsc_avg_position', lambda s: (s > 0).sum()),
             avg_position_first_half=('gsc_avg_position', lambda s: s[s > 0].mean()),
             active_days_first_half=('gsc_impressions', lambda s: (s > 0).sum()))
        .reset_index())

feat = feat[feat['impressions_first_half'] > 0].copy()  # need impressions to compute a CTR at all

# --- Derived numeric feature ---
feat['ctr_first_half'] = feat['clicks_first_half'] / feat['impressions_first_half']

# --- Missing-value handling: avg_position can be NaN if zero days had real position data ---
feat['has_position_data'] = feat['avg_position_first_half'].notna().astype(int)
feat['avg_position_first_half'] = feat['avg_position_first_half'].fillna(feat['avg_position_first_half'].median())
# NOTE: filled with the median + flagged, per flyrank-data's warning against a blind fillna(0)
# injecting a false 'great position' signal - 0 would look like rank 1, which is wrong.

# --- Categorical feature 1: position bucket (one-hot) ---
position_bins = [0, 3, 6, 10, 20, 50, np.inf]
position_labels = ['1-3','4-6','7-10','11-20','21-50','51+']
feat['position_bucket_fh'] = pd.cut(feat['avg_position_first_half'], bins=position_bins, labels=position_labels)
position_dummies = pd.get_dummies(feat['position_bucket_fh'], prefix='pos_bucket')

# --- Categorical feature 2: volume tier (one-hot) ---
vol_bins = feat['impressions_first_half'].quantile([0, .25, .5, .75, 1.0]).values
vol_bins[0] = -1
feat['volume_tier_fh'] = pd.cut(feat['impressions_first_half'], bins=vol_bins,
                                 labels=['Q1_low','Q2','Q3','Q4_high'], duplicates='drop')
volume_dummies = pd.get_dummies(feat['volume_tier_fh'], prefix='volume_tier')

feature_vector = pd.concat([
    feat[['client_hash_id','content_hash_id','clicks_first_half','impressions_first_half',
          'ctr_first_half','avg_position_first_half','has_position_data','active_days_first_half']],
    position_dummies, volume_dummies
], axis=1)

print(f'Feature vector: {len(feature_vector)} rows, {feature_vector.shape[1]} columns.')
feature_vector.head()

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# (a) Column-name / provenance check: every _fh / first_half feature must trace to days<=15 only.
feature_cols = [c for c in feature_vector.columns if c not in ('client_hash_id','content_hash_id')]
suspicious = [c for c in feature_cols if 'second' in c.lower() or 'sh' == c.lower()[-2:] and 'fh' not in c.lower()]
print('Feature columns:', feature_cols)
print('Any column name suggesting second-half/future data:', suspicious if suspicious else 'none')

# (b) ID-as-feature check: pseudonym IDs must never be model inputs.
id_cols_in_features = [c for c in feature_cols if 'hash_id' in c.lower()]
print('ID columns accidentally left in as features:', id_cols_in_features if id_cols_in_features else 'none')

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.